# Traffic Speed Visualization — Duval County

Pulls aggregated speed data from the local FastAPI service and renders it on
a Mapbox choropleth, colored by average speed. Run the API first:

```bash
uv run uvicorn app.main:app --reload
```


In [1]:
# 1. Install dependencies (already installed in this project via `uv sync --group notebook`;
# this line is here for anyone running the notebook outside that environment)
# !pip install requests pandas geopandas mapboxgl shapely python-dotenv

In [2]:
# mapboxgl-jupyter hasn't been updated since ~2019 and imports `display` from a
# location IPython 9 removed it from. Shim it back before importing mapboxgl.
import IPython.core.display
from IPython.display import display

IPython.core.display.display = display

In [3]:
# 2. Setup
import os

import requests
from dotenv import load_dotenv

load_dotenv()

MAPBOX_TOKEN = os.environ["MAPBOX_TOKEN"]
BASE_URL = "http://localhost:8000"

In [4]:
# 3. Request aggregated data
params = {
    "day": "Monday",
    "period": "AM Peak",
}

response = requests.get(f"{BASE_URL}/aggregates/", params=params)
response.raise_for_status()
geojson_data = response.json()
len(geojson_data)

57130

In [ ]:
# 4. Visualize in Mapbox
from mapboxgl.utils import create_color_stops
from mapboxgl.viz import ChoroplethViz

features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "average_speed": f["average_speed"],
            "road_name": f["road_name"],
        },
    }
    for f in geojson_data
]

viz = ChoroplethViz(
    {
        "type": "FeatureCollection",
        "features": features,
    },
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=create_color_stops([10, 20, 30, 40, 50], colors="Reds"),
    center=(-81.6557, 30.3322),
    zoom=11,
    line_width=1.5,
    opacity=0.8,
)
viz.show()

In [6]:
# 5. Tabular summary — slowest 10 segments
import pandas as pd

df = pd.DataFrame(
    [
        {
            "link_id": f["link_id"],
            "avg_speed": f["average_speed"],
            "road_name": f["road_name"],
            "length": f["length"],
        }
        for f in geojson_data
    ]
)
df.sort_values("avg_speed").head(10)

,link_id,avg_speed,road_name,length
11503,23058203,0.621,Toreador Ct,0.042875
50366,1295306964,0.621,Shirley Ave,0.025476
8833,23045422,0.621,Millpoint Dr,0.061516
12821,119046315,0.621,Grady Ct,0.013049
5154,23031339,0.621,11th St N,0.088235
56348,1296146949,0.621,NaN,0.057788
25387,1132265800,0.621,Patriot Ct,0.056545
3522,23025391,0.621,Snow Goose Ln,0.086371
10271,23051359,0.621,Paul Jones Dr,0.052817
8873,23045540,0.621,Buchannan Ct,0.156585
